In [1]:
import pandas as pd
import sqlite3

In [2]:
enrollment = pd.read_csv("../Data/Processed/enrollment_clean.csv")
acute = pd.read_csv("../Data/Processed/acute_care_clean.csv")
eligibility = pd.read_csv("../Data/Processed/eligibility_processing_clean.csv")

conn = sqlite3.connect("../Database/maryland_medicaid.db")

In [3]:
enrollment.to_sql("enrollment", conn, if_exists="replace", index=False)
acute.to_sql("acute_care", conn, if_exists="replace", index=False)
eligibility.to_sql("eligibility_processing", conn, if_exists="replace", index=False)

36

In [4]:
trend = pd.read_sql("""
SELECT
    e.Month,
    e."Total Medicaid Enrollment" AS Medicaid_Enrollment,
    e."Total CHIP Enrollment" AS CHIP_Enrollment,
    e."Total Medicaid and CHIP Enrollment" AS Total_Enrollment,

    p."Beneficiaries with a Renewal Due" AS Renewal_Due,
    p."Beneficiaries Whose Coverage Was Renewed (Total)" AS Renewed,
    p."Beneficiaries Disenrolled at Renewal (Total)" AS Disenrolled,
    p."Beneficiaries Determined Ineligible at Renewal" AS Ineligible,
    p."Beneficiaries Disenrolled for Procedural Reasons at Renewal"
        AS Procedural_Disenrollment,
    p."Beneficiaries with a Pending Renewal" AS Pending_Renewal,

    ROUND(
        100.0 * p."Beneficiaries Whose Coverage Was Renewed (Total)"
        / p."Beneficiaries with a Renewal Due", 2
    ) AS Renewal_Rate,

    ROUND(
        100.0 * p."Beneficiaries Disenrolled at Renewal (Total)"
        / p."Beneficiaries with a Renewal Due", 2
    ) AS Disenrollment_Rate,

    ROUND(
        100.0 * p."Beneficiaries Disenrolled for Procedural Reasons at Renewal"
        / p."Beneficiaries with a Renewal Due", 2
    ) AS Procedural_Disenrollment_Rate,

    ROUND(
        100.0 * p."Beneficiaries Determined Ineligible at Renewal"
        / p."Beneficiaries Disenrolled at Renewal (Total)", 2
    ) AS Ineligible_Share,

    ROUND(
        100.0 * p."Beneficiaries Disenrolled for Procedural Reasons at Renewal"
        / p."Beneficiaries Disenrolled at Renewal (Total)", 2
    ) AS Procedural_Share,

    ROUND(
        100.0 * (
            e."Total Medicaid Enrollment"
            - LAG(e."Total Medicaid Enrollment") OVER (ORDER BY e.Month)
        )
        / LAG(e."Total Medicaid Enrollment") OVER (ORDER BY e.Month),
        2
    ) AS Medicaid_MoM_Percent,

    ROUND(
        100.0 * (
            e."Total CHIP Enrollment"
            - LAG(e."Total CHIP Enrollment") OVER (ORDER BY e.Month)
        )
        / LAG(e."Total CHIP Enrollment") OVER (ORDER BY e.Month),
        2
    ) AS CHIP_MoM_Percent

FROM enrollment e
INNER JOIN eligibility_processing p
    ON e.Month = p.Month

ORDER BY e.Month;
""", conn)

In [5]:
trend.tail()

,Month,Medicaid_Enrollment,CHIP_Enrollment,Total_Enrollment,Renewal_Due,Renewed,Disenrolled,Ineligible,Procedural_Disenrollment,Pending_Renewal,Renewal_Rate,Disenrollment_Rate,Procedural_Disenrollment_Rate,Ineligible_Share,Procedural_Share,Medicaid_MoM_Percent,CHIP_MoM_Percent
31,2025-12-01,1208676.0,198218.0,1406894.0,115503.0,81300.0,32954.0,4444.0,28510.0,1249.0,70.39,28.53,24.68,13.49,86.51,-0.08,-0.00
32,2026-01-01,1204208.0,197098.0,1401306.0,96692.0,70583.0,22472.0,3312.0,19160.0,3637.0,73.00,23.24,19.82,14.74,85.26,-0.37,-0.57
33,2026-02-01,1202579.0,196780.0,1399359.0,89169.0,62231.0,23330.0,2878.0,20452.0,3608.0,69.79,26.16,22.94,12.34,87.66,-0.14,-0.16
34,2026-03-01,1202991.0,196255.0,1399246.0,90024.0,62165.0,23904.0,2802.0,21102.0,3955.0,69.05,26.55,23.44,11.72,88.28,0.03,-0.27
35,2026-04-01,1187556.0,196326.0,1383882.0,89153.0,62471.0,23498.0,3068.0,20430.0,3184.0,70.07,26.36,22.92,13.06,86.94,-1.28,0.04


In [6]:
trend.to_sql(
    "current_trend",
    conn,
    if_exists="replace",
    index=False
)

36

In [7]:
current_analysis = pd.read_sql("""
SELECT
    *,
    LEAD(Medicaid_MoM_Percent, 1)
        OVER (ORDER BY Month)
        AS Next_Month_Medicaid_MoM_Percent
FROM current_trend
ORDER BY Month;
""", conn)

In [8]:
current_analysis[
    [
        "Medicaid_MoM_Percent",
        "Next_Month_Medicaid_MoM_Percent",
        "Renewal_Rate",
        "Disenrollment_Rate",
        "Procedural_Disenrollment_Rate",
        "Procedural_Disenrollment"
    ]
].dropna().corr()

,Medicaid_MoM_Percent,Next_Month_Medicaid_MoM_Percent,Renewal_Rate,Disenrollment_Rate,Procedural_Disenrollment_Rate,Procedural_Disenrollment
Medicaid_MoM_Percent,1.000000,0.425521,0.481944,-0.487654,-0.406221,-0.488239
Next_Month_Medicaid_MoM_Percent,0.425521,1.000000,0.671482,-0.649269,-0.581417,-0.744772
Renewal_Rate,0.481944,0.671482,1.000000,-0.990806,-0.953065,-0.895226
Disenrollment_Rate,-0.487654,-0.649269,-0.990806,1.000000,0.969039,0.912822
Procedural_Disenrollment_Rate,-0.406221,-0.581417,-0.953065,0.969039,1.000000,0.908265
Procedural_Disenrollment,-0.488239,-0.744772,-0.895226,0.912822,0.908265,1.000000


In [9]:
historical = pd.read_sql("""
SELECT
    e.Month,
    e."Total Medicaid Enrollment" AS Medicaid_Enrollment,
    e."Total CHIP Enrollment" AS CHIP_Enrollment,

    MAX(CASE
        WHEN a.AcuteCareService = 'ED visits'
        THEN a.ServiceCount
    END) AS ED_Visits,

    MAX(CASE
        WHEN a.AcuteCareService = 'ED visits'
        THEN a.RatePer1000Beneficiaries
    END) AS ED_Rate_Per_1000,

    MAX(CASE
        WHEN a.AcuteCareService = 'Inpatient stays'
        THEN a.ServiceCount
    END) AS Inpatient_Stays,

    MAX(CASE
        WHEN a.AcuteCareService = 'Inpatient stays'
        THEN a.RatePer1000Beneficiaries
    END) AS Inpatient_Rate_Per_1000

FROM enrollment e
INNER JOIN acute_care a
    ON e.Month = a.Month

GROUP BY
    e.Month,
    e."Total Medicaid Enrollment",
    e."Total CHIP Enrollment"

ORDER BY e.Month;
""", conn)

In [10]:
historical["Medicaid_MoM_Percent"] = (
    historical["Medicaid_Enrollment"].pct_change() * 100
)

historical["ED_MoM_Percent"] = (
    historical["ED_Rate_Per_1000"].pct_change() * 100
)

historical["Inpatient_MoM_Percent"] = (
    historical["Inpatient_Rate_Per_1000"].pct_change() * 100
)

In [11]:
historical[
    [
        "Medicaid_MoM_Percent",
        "ED_MoM_Percent",
        "Inpatient_MoM_Percent"
    ]
].dropna().corr()

,Medicaid_MoM_Percent,ED_MoM_Percent,Inpatient_MoM_Percent
Medicaid_MoM_Percent,1.00000,-0.135750,-0.093800
ED_MoM_Percent,-0.13575,1.000000,0.632828
Inpatient_MoM_Percent,-0.09380,0.632828,1.000000


In [12]:
for lag in [1, 2, 3]:
    historical[f"ED_Lag{lag}"] = historical["ED_MoM_Percent"].shift(-lag)
    historical[f"Inpatient_Lag{lag}"] = historical["Inpatient_MoM_Percent"].shift(-lag)

In [13]:
historical[
    [
        "Medicaid_MoM_Percent",
        "ED_Lag1",
        "ED_Lag2",
        "ED_Lag3",
        "Inpatient_Lag1",
        "Inpatient_Lag2",
        "Inpatient_Lag3"
    ]
].corr()["Medicaid_MoM_Percent"]

Medicaid_MoM_Percent    1.000000
ED_Lag1                 0.003080
ED_Lag2                 0.005842
ED_Lag3                 0.015816
Inpatient_Lag1         -0.064520
Inpatient_Lag2         -0.121381
Inpatient_Lag3         -0.062105
Name: Medicaid_MoM_Percent, dtype: float64

In [14]:
trend.to_sql(
    "current_analysis",
    conn,
    if_exists="replace",
    index=False
)

historical.to_sql(
    "historical_utilization",
    conn,
    if_exists="replace",
    index=False
)

60